# Train SVM

Development data and features are temporary until Role D provides the shared versions. Training requires W&B unless `train.require_wandb` is explicitly disabled for a local smoke test.


## 0) Bootstrap (Kaggle / Colab)

Run once per session: clone or pull this repo, `cd` into it, install deps. Locally you can skip clone if you already opened the notebook inside the repo.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Classification.git"
REPO_DIR = "VN-Traffic-Sign-Classification"

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str) -> None:
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

if (repo_path / ".git").exists():
    sh(f'git -C "{repo_path}" pull --ff-only')
elif (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using existing repo at", repo_path)
    try:
        sh("git pull --ff-only")
    except subprocess.CalledProcessError:
        print("git pull skipped/failed — continue with local tree")
else:
    sh(f'git clone {REPO_URL} "{repo_path}"')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

req = repo_path / "requirements-kaggle.txt"
if req.exists():
    sh(f'pip install -q -r "{req}"')
else:
    sh("pip install -q -r " + str(repo_path / "requirements.txt"))
sh("pip install -q -e .")
print("bootstrap OK")


## 1) Resolve config and locate processed data

Set `SVM_PROCESSED_ROOT` when the dataset is outside the repository. Use `dev_preprocess` first if metadata is absent.


In [ ]:
import copy
import os
from pathlib import Path
from vn_tsc.config.resolve import resolve_config

cfg = resolve_config('configs/pipelines/svm.yaml', 'configs/shared.yaml', 'configs/runtime/kaggle_2xt4.yaml')
cfg['data_source']['processed_root'] = os.environ.get('SVM_PROCESSED_ROOT', cfg['data_source']['processed_root'])
processed_root = Path(cfg['data_source']['processed_root'])
metadata_path = processed_root / 'metadata.csv'
if not metadata_path.is_file():
    raise FileNotFoundError(f'{metadata_path} missing. Run: python -m vn_tsc.pipelines.svm.dev_preprocess --data-root <RAW_PATH>')
print('metadata:', metadata_path)
print('feature backend:', cfg['features']['backend'])


## 2) Build or load feature cache and inspect shapes

The adapter follows train/val/test assignments in metadata. Feature caching is controlled by `features.cache`.


In [ ]:
from vn_tsc.pipelines.svm.data_adapter import load_svm_data

data = load_svm_data(cfg)
print('train:', data.X_train.shape, data.y_train.shape)
print('val:  ', data.X_val.shape, data.y_val.shape)
print('test: ', data.X_test.shape, data.y_test.shape)
print('classes:', data.class_names)


## 3) Train baseline SVM

The CLI handles fitting, validation, artifacts, and W&B logging. Test evaluation remains disabled during model selection.


In [ ]:
import json
import subprocess
import sys
import tempfile
import yaml
from IPython.display import Image as NotebookImage, display

def run_experiment(config, name):
    with tempfile.TemporaryDirectory() as temp:
        config_path = Path(temp) / 'svm_experiment.yaml'
        config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
        subprocess.run([sys.executable, '-m', 'tools.run_train', '--pipeline', 'svm',
                        '--config', str(config_path), '--run-name', name], check=True)
    candidates = list((Path(config['outputs']['root']) / 'svm').glob(f'*_{name}'))
    return max(candidates, key=lambda path: path.stat().st_mtime)

baseline_cfg = copy.deepcopy(cfg)
baseline_cfg['dim_reduction']['method'] = 'none'
baseline_cfg['evaluation']['evaluate_test_after_train'] = False
baseline_run = run_experiment(baseline_cfg, 'baseline_dev')
print('run:', baseline_run)


## 4) Validation metrics and confusion matrix


In [ ]:
baseline_metrics = json.loads((baseline_run / 'metrics.json').read_text(encoding='utf-8'))
print('validation:', baseline_metrics['validation'])
figure = baseline_run / 'figures' / 'confusion_matrix_val.png'
if figure.exists():
    display(NotebookImage(filename=str(figure)))


## 5) PCA experiment

PCA remains inside the sklearn Pipeline. Choose `n_components` as a variance fraction or a valid integer.


In [ ]:
RUN_EXTRA_EXPERIMENTS = False
pca_cfg = copy.deepcopy(baseline_cfg)
pca_cfg['dim_reduction']['method'] = 'pca'
pca_cfg['dim_reduction']['pca']['n_components'] = 0.95
if RUN_EXTRA_EXPERIMENTS:
    pca_run = run_experiment(pca_cfg, 'pca_dev')
    print(json.loads((pca_run / 'metrics.json').read_text())['validation'])


## 6) LDA experiment

LDA uses labels from the training split and is fitted inside the sklearn Pipeline.


In [ ]:
lda_cfg = copy.deepcopy(baseline_cfg)
lda_cfg['dim_reduction']['method'] = 'lda'
lda_cfg['dim_reduction']['lda']['n_components'] = None
if RUN_EXTRA_EXPERIMENTS:
    lda_run = run_experiment(lda_cfg, 'lda_dev')
    print(json.loads((lda_run / 'metrics.json').read_text())['validation'])


## 7) Optional tuning

CV uses training data only; validation scores are reported after choosing parameters.


In [ ]:
tuning_cfg = copy.deepcopy(baseline_cfg)
tuning_cfg['tuning']['enabled'] = True
tuning_cfg['tuning']['method'] = 'grid'  # or random
tuning_cfg['tuning']['param_grid'] = {'svm__C': [0.1, 1.0, 10.0], 'svm__gamma': ['scale', 'auto']}
if RUN_EXTRA_EXPERIMENTS:
    tuning_run = run_experiment(tuning_cfg, 'tuning_dev')
    print(json.loads((tuning_run / 'metrics.json').read_text())['validation'])


## 8) W&B and final evaluation

Open the W&B project configured by `project.wandb_entity` and `project.wandb_project` to compare development runs. Once a configuration is frozen, set `evaluation.evaluate_test_after_train: true` and run it once for final test metrics. Development backend results are labeled temporary.


In [ ]:
print('W&B:', cfg['project']['wandb_entity'], '/', cfg['project']['wandb_project'])
print('model:', baseline_run / 'checkpoints' / 'model.joblib')
print('feature config:', baseline_run / 'metadata' / 'feature_config.json')
